# Carbon intensity forecasting: a reproducible modelling demo

This notebook shows how to turn time-stamped grid carbon intensity into a small, testable forecasting problem. It uses a deterministic sample when the live API is unavailable, so the notebook remains runnable offline.

In [ ]:
from datetime import timezone
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

def make_sample(days=21, seed=7):
    rng = np.random.default_rng(seed)
    timestamps = pd.date_range('2025-01-01', periods=days * 24, freq='h', tz=timezone.utc)
    hour = timestamps.hour.to_numpy()
    day = timestamps.dayofweek.to_numpy()
    intensity = (180 + 45*np.sin((hour - 7) * np.pi / 12)
                 + 18*np.cos(day * 2*np.pi / 7) + rng.normal(0, 10, len(timestamps)))
    return pd.DataFrame({'timestamp': timestamps, 'carbon_intensity': intensity.round(1)})

df = make_sample()
df.head()

## 1. Create features from the timestamp

Operational forecasting often starts with surprisingly useful calendar signals. We also add lagged observations, then drop the first rows where those lags do not exist.

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].ge(5).astype(int)
df['lag_1h'] = df['carbon_intensity'].shift(1)
df['lag_24h'] = df['carbon_intensity'].shift(24)
df['rolling_24h'] = df['carbon_intensity'].shift(1).rolling(24).mean()
df = df.dropna().reset_index(drop=True)
features = ['hour', 'day_of_week', 'is_weekend', 'lag_1h', 'lag_24h', 'rolling_24h']

## 2. Evaluate in time order

The final 20% is held out as a future period. This mirrors how the model would be used in practice and avoids leaking future measurements into training.

In [ ]:
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
X_train, y_train = train[features], train['carbon_intensity']
X_test, y_test = test[features], test['carbon_intensity']

baseline = test['lag_24h']
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
pred = model.predict(X_test)

results = pd.DataFrame([
    {'model': 'same hour yesterday', 'MAE': mean_absolute_error(y_test, baseline), 'RMSE': np.sqrt(mean_squared_error(y_test, baseline))},
    {'model': 'ridge regression', 'MAE': mean_absolute_error(y_test, pred), 'RMSE': np.sqrt(mean_squared_error(y_test, pred))},
]).sort_values('RMSE')
results

## 3. Practical interpretation

A useful data product would expose both the forecast and its uncertainty, log the data timestamp used for each prediction, and monitor drift when generation mix or weather patterns change. The key lesson is the workflow: acquire, validate, engineer, compare with a baseline, and evaluate chronologically.